# Week 5 Solution — Advanced RAG for Insurellm

This notebook demonstrates the full improved pipeline and runs evaluation against the test suite.

## What's improved vs the baseline (`implementation/`)

| Area | Baseline | This Solution |
|------|----------|---------------|
| Embeddings | `all-MiniLM-L6-v2` (384-dim) | `text-embedding-3-large` (3072-dim) |
| Chunking | `RecursiveCharacterTextSplitter` | LLM semantic chunking (headline + summary + original) |
| Chunk size | 500 chars | ~150 chars avg → fine-grained per-fact chunks |
| Retrieval | Single query, top-10 | Dual retrieval (original + rewritten query), merged pool |
| Reranking | None | LLM reranker on merged candidate pool |
| Prompt | Basic | Tuned for accuracy + completeness + relevance |

**Run order:**
1. Run `ingest.py` once to build the vector store → `week5_solution_db/`
2. Use this notebook to test individual questions and run full evaluation

In [ ]:
import sys, os
# Make sure the week5/ directory is on the path when running from week5/week5_solution/
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(override=True)

## Step 1 — Build the vector store

Run this once. It will:
- Load all markdown docs from `knowledge-base/`
- Use `gpt-4.1-nano` to split each document into fine-grained, semantically meaningful chunks
- Embed every chunk with `text-embedding-3-large`
- Persist to `week5_solution_db/`

In [ ]:
from week5_solution.ingest import fetch_documents, create_chunks, create_embeddings

documents = fetch_documents()
print(f"Documents loaded: {len(documents)}")

In [ ]:
chunks = create_chunks(documents)
print(f"Chunks created: {len(chunks)}")

In [ ]:
create_embeddings(chunks)

## Step 2 — Inspect a sample chunk

Each chunk has three layers:
- **Headline** — entity-rich heading optimised for keyword matching
- **Summary** — dense restatement of every specific fact in the chunk
- **Original text** — verbatim source text

In [ ]:
print(chunks[0].page_content)
print("---")
print(f"Source: {chunks[0].metadata}")

## Step 3 — Test individual questions

In [ ]:
from week5_solution.answer import answer_question, fetch_context

question = "Who won the IIOTY award in 2023?"
answer, retrieved = answer_question(question)
print("Answer:\n", answer)

In [ ]:
print("\nRetrieved chunks:")
for i, chunk in enumerate(retrieved, 1):
    print(f"\n--- Chunk {i} [{chunk.metadata.get('source','?')}] ---")
    print(chunk.page_content[:300], "...")

In [ ]:
# Spanning question — requires facts from multiple documents
q2 = "What is the salary of the CTO who joined in January 2017?"
ans2, _ = answer_question(q2)
print(ans2)

In [ ]:
q3 = "What are Insurellm's four core values?"
ans3, _ = answer_question(q3)
print(ans3)

## Step 4 — Evaluate a single test case

In [ ]:
from week5_solution.eval import evaluate_retrieval, evaluate_answer
from week5_solution.test import load_tests

tests = load_tests()
print(f"Total test cases: {len(tests)}")

In [ ]:
# Evaluate test #0
test = tests[0]
print(f"Question : {test.question}")
print(f"Reference: {test.reference_answer}")

r = evaluate_retrieval(test)
print(f"\nRetrieval — MRR: {r.mrr:.4f}  nDCG: {r.ndcg:.4f}  Coverage: {r.keyword_coverage:.1f}%")

a, gen, _ = evaluate_answer(test)
print(f"\nGenerated answer:\n{gen}")
print(f"\nFeedback: {a.feedback}")
print(f"Accuracy={a.accuracy}/5  Completeness={a.completeness}/5  Relevance={a.relevance}/5")

## Step 5 — Full evaluation run

This iterates over all 150 test cases and computes aggregate scores.
Takes ~10-15 minutes due to API calls.

In [ ]:
from week5_solution.eval import evaluate_all_answers, evaluate_all_retrieval
from tqdm.notebook import tqdm

answer_results = []
for test, result, progress in tqdm(evaluate_all_answers(), total=len(tests)):
    answer_results.append((test, result))

total = len(answer_results)
avg_acc  = sum(r.accuracy     for _, r in answer_results) / total
avg_comp = sum(r.completeness for _, r in answer_results) / total
avg_rel  = sum(r.relevance    for _, r in answer_results) / total

print(f"\n{'='*50}")
print(f"ANSWER QUALITY (n={total})")
print(f"Mean Accuracy    : {avg_acc:.4f} / 5")
print(f"Mean Completeness: {avg_comp:.4f} / 5")
print(f"Mean Relevance   : {avg_rel:.4f} / 5")
print(f"{'='*50}")

In [ ]:
retrieval_results = []
for test, result, progress in tqdm(evaluate_all_retrieval(), total=len(tests)):
    retrieval_results.append((test, result))

avg_mrr  = sum(r.mrr              for _, r in retrieval_results) / total
avg_ndcg = sum(r.ndcg             for _, r in retrieval_results) / total
avg_cov  = sum(r.keyword_coverage for _, r in retrieval_results) / total

print(f"\n{'='*50}")
print(f"RETRIEVAL QUALITY (n={total})")
print(f"Mean MRR         : {avg_mrr:.4f}")
print(f"Mean nDCG        : {avg_ndcg:.4f}")
print(f"Mean Kw Coverage : {avg_cov:.2f}%")
print(f"{'='*50}")

## Step 6 — Interactive chat demo

In [ ]:
history = []

def chat(question: str):
    global history
    answer, _ = answer_question(question, history)
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})
    print(f"Q: {question}")
    print(f"A: {answer}\n")

chat("Who founded Insurellm and when?")
chat("What is their current salary?")
chat("What was the company's first product?")